# WorldSim 2B Model — Training + GGUF Conversion
Fine-tune Qwen3.5-2B-Base on v2 dataset, evaluate, convert to GGUF Q4_K_M


## 1. Environment & Dataset Verification


In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd()
repo_marker = cwd / 'training/run_qlora_train.py'
notebook_marker = cwd / 'notebooks/dgx_spark_2b_train_and_convert.ipynb'
assert repo_marker.exists() and notebook_marker.exists(), (
    'Run this notebook from the repository root: worldsim-training'
)
REPO_ROOT = cwd
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

MODEL_2B = 'Qwen/Qwen3.5-2B-Base'
DATASET_ID = 'worldsim-2b-v2-mix'
OUTPUT_BASE = REPO_ROOT / 'outputs' / 'baseline' / DATASET_ID

V2_TRAINING_DIR = REPO_ROOT / 'data' / 'training' / 'worldsim-v2-mix'
train_file = V2_TRAINING_DIR / 'train_curriculum.jsonl'
dev_file = V2_TRAINING_DIR / 'dev_converted.jsonl'
assert train_file.exists(), f'Training data not found: {train_file}'
assert dev_file.exists(), f'Dev data not found: {dev_file}'

with train_file.open(encoding='utf-8') as handle:
    train_count = sum(1 for _ in handle)
with dev_file.open(encoding='utf-8') as handle:
    dev_count = sum(1 for _ in handle)

LLAMA_CPP_DIR = REPO_ROOT / 'tools' / 'llama.cpp'
LLAMA_QUANTIZE = LLAMA_CPP_DIR / 'build' / 'bin' / 'llama-quantize'
LLAMA_COMPLETION = LLAMA_CPP_DIR / 'build' / 'bin' / 'llama-completion'
CONVERT_SCRIPT = LLAMA_CPP_DIR / 'convert_hf_to_gguf.py'
GGUF_OUTPUT_DIR = REPO_ROOT / 'artifacts' / 'gguf'
GGUF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Model:       {MODEL_2B}')
print(f'Dataset ID:  {DATASET_ID}')
print(f'Train file:  {train_file} ({train_count} samples)')
print(f'Dev file:    {dev_file} ({dev_count} samples)')
print(f'Output base: {OUTPUT_BASE}')
print(f'llama.cpp:   {'✅' if LLAMA_QUANTIZE.exists() and LLAMA_COMPLETION.exists() else '❌'}')
print(f'GGUF output: {GGUF_OUTPUT_DIR}')
print('Environment ready ✅')


## 2. Training Config


In [ ]:
from datetime import UTC, datetime

from training.lib.qlora_smoke import resolve_baseline_notebook_config

RUN_ID = datetime.now(UTC).strftime('run-%Y%m%dT%H%M%SZ')
OUTPUT_DIR = OUTPUT_BASE / RUN_ID

CONFIG = resolve_baseline_notebook_config(
    RUN_ID,
    output_dir_override=OUTPUT_DIR,
    overrides={
        'model_name': MODEL_2B,
        'dry_run': False,
        'require_qlora': True,
        'dataset': DATASET_ID,
        'train_file': str(train_file),
        'dev_file': str(dev_file),
        'max_train_samples': 0,
        'max_eval_samples': 0,
        'max_steps': 1296,
        'gradient_accumulation_steps': 8,
        'learning_rate': 1e-4,
        'logging_steps': 10,
        'eval_steps': 100,
        'save_steps': 100,
        'save_total_limit': 3,
        'per_device_train_batch_size': 1,
        'per_device_eval_batch_size': 1,
    },
)

print(f'Run ID:      {RUN_ID}')
print(f'Model:       {CONFIG["model_name"]}')
print(f'Output:      {CONFIG["output_dir"]}')
print(f'Max steps:   {CONFIG["max_steps"]}')
print(f'LR:          {CONFIG["learning_rate"]}')
print('Config ready ✅')


## 3. Train


In [ ]:
import time
from pathlib import Path

from training.lib.qlora_smoke import SmokeRunConfig, run_baseline_or_raise

cfg = SmokeRunConfig(
    run_mode='baseline',
    model_name=CONFIG['model_name'],
    train_file=CONFIG['train_file'],
    dev_file=CONFIG['dev_file'],
    output_dir=CONFIG['output_dir'],
    max_steps=CONFIG['max_steps'],
    max_train_samples=CONFIG['max_train_samples'],
    max_eval_samples=CONFIG['max_eval_samples'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_eval_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    logging_steps=CONFIG['logging_steps'],
    eval_steps=CONFIG['eval_steps'],
    save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG.get('save_total_limit', 1),
    require_qlora=CONFIG['require_qlora'],
    seed=CONFIG['seed'],
    dry_run=CONFIG['dry_run'],
)

started = time.perf_counter()
result = run_baseline_or_raise(cfg)
elapsed = time.perf_counter() - started

print()
print('=' * 60)
print('  2B TRAINING COMPLETE')
print('=' * 60)
print(f'  elapsed_min: {(elapsed / 60):.1f}')
print(f'  output_dir:  {result.output_dir}')
print(f'  adapter_dir: {Path(result.output_dir) / "adapter"}')
print(f'  train_loss:  {result.train_loss}')
print(f'  eval_loss:   {result.eval_loss}')
print(f'  status:      {result.status}')


## 4. Guardrail Evaluation


In [ ]:
from training.lib.qlora_smoke import (
    load_json_artifact,
    load_optional_json_artifact,
    load_sample_generations,
    summarize_sample_generations,
)

run_config_artifact = load_json_artifact(result.output_dir, 'run_config.json')
summary_artifact = load_json_artifact(result.output_dir, 'summary.json')
metrics_artifact = load_optional_json_artifact(result.output_dir, 'metrics.json')
sample_rows = load_sample_generations(result.output_dir)
sample_summary = summarize_sample_generations(sample_rows) if sample_rows else None
structured_metrics = (metrics_artifact or {}).get('structured_metrics', {})

print()
print('=' * 60)
print('  2B GUARDRAIL RESULTS')
print('=' * 60)
for key in [
    'training_completed',
    'train_loss',
    'eval_loss',
    'output_dir',
    'adapter_dir',
]:
    print(f'  {key}: {summary_artifact.get(key, "N/A")}')

if structured_metrics:
    for key in [
        'per_sample_success_rate',
        'structured_success_rate',
        'json_parse_failure_rate',
        'repair_applied_rate',
        'extra_key_rate',
    ]:
        value = structured_metrics.get(key, 'N/A')
        print(f"  {key}: {f'{value:.4f}' if isinstance(value, float) else value}")

if sample_summary:
    print(f"  sample_count: {sample_summary.get('total_samples')}")
    print(f"  malformed_json: {sample_summary.get('malformed_json_count')}")
    print(f"  language_drift: {sample_summary.get('language_drift_count')}")


## 5. LoRA Merge → BF16


In [ ]:
import torch
from pathlib import Path

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

adapter_dir = Path(CONFIG['output_dir']) / 'adapter'
merged_dir = Path(CONFIG['output_dir']) / 'merged_bf16'

if merged_dir.exists() and (merged_dir / 'config.json').exists():
    print(f'Already merged at {merged_dir}, skipping... ✅')
else:
    print(f'Loading base model {MODEL_2B} in BF16...')
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_2B,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=False,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_2B, trust_remote_code=False)

    print('Loading LoRA adapter...')
    model = PeftModel.from_pretrained(base_model, str(adapter_dir))

    print('Merging...')
    merged_model = model.merge_and_unload()

    merged_dir.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(str(merged_dir))
    tokenizer.save_pretrained(str(merged_dir))

    del merged_model, model, base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    size_gb = sum(path.stat().st_size for path in merged_dir.rglob('*') if path.is_file()) / 1e9
    print(f'Merged model saved ({size_gb:.2f} GB) ✅')


## 6. Convert HF → GGUF (FP16)


In [ ]:
import subprocess
from pathlib import Path

fp16_gguf = Path(CONFIG['output_dir']) / 'worldsim-v2-qwen3.5-2b-f16.gguf'

if fp16_gguf.exists():
    print(f'FP16 GGUF exists: {fp16_gguf} ({fp16_gguf.stat().st_size / 1e9:.2f} GB) ✅')
else:
    print('Converting HF → GGUF (FP16)...')
    result_convert = subprocess.run(
        [
            sys.executable,
            str(CONVERT_SCRIPT),
            str(merged_dir),
            '--outfile',
            str(fp16_gguf),
            '--outtype',
            'f16',
        ],
        capture_output=True,
        text=True,
    )
    if result_convert.returncode != 0:
        print(f'STDERR: {result_convert.stderr[-2000:]}')
        raise RuntimeError('GGUF conversion failed')
    print(f'FP16 GGUF created: {fp16_gguf.stat().st_size / 1e9:.2f} GB ✅')


## 7. Quantize → Q4_K_M


In [ ]:
from pathlib import Path

q4_gguf = Path(CONFIG['output_dir']) / 'worldsim-v2-qwen3.5-2b-q4_k_m.gguf'

if q4_gguf.exists():
    print(f'Q4_K_M GGUF exists: {q4_gguf} ({q4_gguf.stat().st_size / 1e6:.0f} MB) ✅')
else:
    print('Quantizing FP16 → Q4_K_M...')
    result_quant = subprocess.run(
        [str(LLAMA_QUANTIZE), str(fp16_gguf), str(q4_gguf), 'Q4_K_M'],
        capture_output=True,
        text=True,
    )
    if result_quant.returncode != 0:
        print(f'STDERR: {result_quant.stderr[-2000:]}')
        raise RuntimeError('Quantization failed')
    print(f'Q4_K_M GGUF created: {q4_gguf.stat().st_size / 1e6:.0f} MB ✅')


## 8. Verify GGUF


In [ ]:
import json
import re

print('=== 2B GGUF Verification ===')
print(f'File: {q4_gguf}')
print(f'Size: {q4_gguf.stat().st_size / 1e6:.0f} MB')


def strip_think(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


print()
print('--- Test 1: JSON generation ---')
test_prompt = '''[TASK] E
[TEMP]
NS=0.8 HA=0.2 RD=0.5 P=0.7 type=choleric
[personality]
bold, impulsive
[situation]
stranger group approaching
[options]
0:flee 1:hide 2:confront 3:warn
[output]
{"action_id": number, "confidence": 0.0-1.0, "hint_en": "one sentence"}'''

result_json = subprocess.run(
    [
        str(LLAMA_COMPLETION),
        '-m', str(q4_gguf),
        '-p', test_prompt,
        '-n', '128',
        '--temp', '0',
        '-t', '4',
        '-ngl', '99',
        '--no-display-prompt',
    ],
    capture_output=True,
    text=True,
    timeout=120,
)
output = strip_think(result_json.stdout.strip())
print(f'Output: {output[:500]}')
try:
    parsed = json.loads(output.splitlines()[0])
    print(f'Valid JSON ✅: {parsed}')
except Exception as exc:
    print(f'Parse issue ⚠️: {exc}')

print()
print('--- Test 2: Korean narrative ---')
test_ko = '''[TASK] B
[TEMP]
NS=0.2 HA=0.8 RD=0.6 P=0.4 type=melancholic
[personality]
cautious, anxious
[emotion]
fear:0.8
[situation]
predator spotted near camp'''

result_ko = subprocess.run(
    [
        str(LLAMA_COMPLETION),
        '-m', str(q4_gguf),
        '-p', test_ko,
        '-n', '150',
        '--temp', '0.7',
        '-t', '4',
        '-ngl', '99',
        '--no-display-prompt',
    ],
    capture_output=True,
    text=True,
    timeout=120,
)
output_ko = strip_think(result_ko.stdout.strip())
print(f'Output: {output_ko[:500]}')


## 9. Copy to artifacts & Summary


In [ ]:
import shutil

final_gguf = GGUF_OUTPUT_DIR / 'worldsim-v2-qwen3.5-2b-q4_k_m.gguf'
shutil.copy2(q4_gguf, final_gguf)

gguf_08b = GGUF_OUTPUT_DIR / 'worldsim-v2-qwen3.5-0.8b-q4_k_m.gguf'

print('=' * 60)
print('  2B TRAINING + GGUF CONVERSION COMPLETE')
print('=' * 60)
print(f'  Model:        {MODEL_2B}')
print(f'  Dataset:      {train_count} train, {dev_count} dev')
print(f'  train_loss:   {summary_artifact.get("train_loss", "N/A")}')
print(f'  eval_loss:    {summary_artifact.get("eval_loss", "N/A")}')
print(f'  Adapter:      {adapter_dir}')
print(f'  FP16 GGUF:    {fp16_gguf} ({fp16_gguf.stat().st_size / 1e6:.0f} MB)')
print(f'  Q4_K_M GGUF:  {final_gguf} ({final_gguf.stat().st_size / 1e6:.0f} MB)')
print()
print('  === DUAL MODEL STATUS ===')
print(f"  0.8B GGUF:    {'✅ ' + str(gguf_08b) + f' ({gguf_08b.stat().st_size / 1e6:.0f} MB)' if gguf_08b.exists() else '❌ not found'}")
print(f'  2B GGUF:      ✅ {final_gguf} ({final_gguf.stat().st_size / 1e6:.0f} MB)')
if gguf_08b.exists():
    combined_mb = (gguf_08b.stat().st_size + final_gguf.stat().st_size) / 1e6
    print(f'  Combined:     ~{combined_mb:.0f} MB')
print('=' * 60)


## 10. Next: Re-run Parallel Benchmark


In [ ]:
print('=' * 60)
print('  NEXT STEPS')
print('=' * 60)
print()
print('  1. Re-run parallel benchmark notebook:')
print('     notebooks/dgx_spark_parallel_benchmark.ipynb')
print('     → 2B GGUF now exists, Experiment 4 will auto-include 2B tests')
print()
print('  2. Compare 0.8B vs 2B quality:')
print('     Same prompts, both models → per-task success rate comparison')
print()
print('  Dual model files ready in:')
print(f'     {GGUF_OUTPUT_DIR}/')
print('=' * 60)
